# Find enriched SAE features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/feature_enrichment.ipynb)

Compare a positive sequence set with a length-matched background. Outputs: input audits,
background diagnostics, a complete enrichment table, activation-window logos, and an optional GRPO signature.
The default uses nuclear pore complex IDRs and a sampled validation background.

Run cells from top to bottom. In Colab select **Runtime → Change runtime type → GPU**.
A GPU is recommended; CPU works but is slower. Runtime and peak memory depend on sequence
length, model, and hardware; timings are printed below rather than promising a fixed runtime.
First use downloads model weights. Outputs are written under `OUT_DIR`; rerunning replaces
files with the same names. Download that folder from Colab before ending the session.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git@v1"])

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.io import Record, read_fasta, parse_idr_header
print("Python:", sys.version.split()[0])
import idiom
print("IDiom:", idiom.__file__)


# Locate the companion helper in a clone, or download it for standalone Colab use.
helper_dir = next((p for p in (Path.cwd(), Path.cwd() / "cookbook/notebooks")
                   if (p / "workflow_utils.py").is_file()), None)
if helper_dir is None:
    from urllib.request import urlretrieve
    helper_dir = Path(".idiom_notebook_helpers")
    helper_dir.mkdir(exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/rotskoff-group/idiom/main/"
                "cookbook/notebooks/workflow_utils.py", helper_dir / "workflow_utils.py")
sys.path.insert(0, str(helper_dir.resolve()))
from workflow_utils import (AA, DEMO, load_inputs, idr_sequence, isolated, check_context,
                            summaries, write_fasta, save_run)


## Choose the comparison before running

Set `POSITIVE_FASTA` and `BACKGROUND_FASTA` to your own files. An appropriate background represents
the alternative you want to compare against; a broad database answers a different question from
other IDRs measured in the same assay. Length matching does not control species, homology,
composition, or assay selection. Exact duplicate IDRs are collapsed within each input, and all
positive IDRs are excluded from the background before sampling. Related sequences can still remain.

The default downloads the validation FASTA but encodes only a small sample. These defaults check
the workflow and do not reproduce released signatures. Increase sample sizes for your analysis.


In [ ]:
POSITIVE_FASTA = None
BACKGROUND_FASTA = None
POSITIVE_MODE = "idr" # Default NPC examples are isolated IDRs
BACKGROUND_MODE = "annotated" # Default validation records contain annotated spans
NAME = "nuclear_pore_complex"
SAE = "jxliu2/idiomsae-300M-L18-k32"
DEVICE = "auto"
OUT_DIR = Path("feature_enrichment_outputs")
MAX_POSITIVE = 128
MAX_BACKGROUND = 512
BATCH_SIZE = 2
TOP_N = 30
CASE = f"top{TOP_N}"
SEED = 0
N_FEATURES = 9
N_WINDOWS = 60
HALF_WIDTH = 7
EXPORT_SIGNATURE = False # Optional handoff to GRPO


## Inputs and validation

Use `POSITIVE_MODE="idr"` / `BACKGROUND_MODE="idr"` for FASTA records that are **already isolated IDRs** (ordinary headers
are accepted). Use `"annotated"` mode for full proteins: the first header token must
end in `_IDR_x-y`, with **1-based inclusive** coordinates. This notebook does not predict IDR
boundaries. Python slices use 0-based, end-exclusive coordinates.

The default inputs are the NPC example and validation background described above.
Set local FASTA paths to analyze your own files (upload it using the Colab Files pane).
The audit table reports rejected records and records outside the sample limit. Empty sequences,
noncanonical residues, and invalid annotations are not silently repaired. Repeated accessions
remain distinct through `record_id`; duplicate IDR sequences are reported for your review.


In [ ]:
started = time.perf_counter()


In [ ]:
from huggingface_hub import hf_hub_download
from idiom.sae.features import FeatureDataset
from idiom.sae.features.enrichment import (
    FDR_ALPHA, LOG2OR_FLOOR, PREV_POS_FLOOR, enrich, enriched_mask,
    feature_counts, length_match, top_features, write_signature,
)
out = OUT_DIR
out.mkdir(parents=True, exist_ok=True)
if MAX_BACKGROUND < 1 or TOP_N < 1 or BATCH_SIZE < 1 or (MAX_POSITIVE is not None and MAX_POSITIVE < 1):
    raise ValueError("Sample limits, TOP_N, and BATCH_SIZE must be positive.")
positive_path = Path(POSITIVE_FASTA) if POSITIVE_FASTA is not None else Path(hf_hub_download(
    "jxliu2/idiom-db", "other/example_data/protgps/nuclear_pore_complex.fasta", repo_type="dataset"))
bg_path = Path(BACKGROUND_FASTA) if BACKGROUND_FASTA is not None else Path(hf_hub_download(
    "jxliu2/idiom-db", "idiom-db/idiom-db-v1_validation.fasta", repo_type="dataset"))
sae = IDiomSAE.from_pretrained(SAE, device=DEVICE)
if sae.fim_mode != "unprompted" or sae.region != "idr":
    raise ValueError("Use an unprompted IDR SAE.")

def unique_usable(path, mode, label):
    recs, audit = load_inputs(path, mode, None)
    seen, kept, reasons = set(), [], {}
    for r in recs:
        s = idr_sequence(r)
        reason = None
        if len(s) + 4 > sae.model.cfg.max_seq_len:
            reason = "exceeds model context"
        elif s in seen:
            reason = "duplicate IDR"
        else:
            seen.add(s)
            kept.append(r)
        if reason:
            reasons[r.accession] = reason
    audit["status"] = audit.record_id.map(reasons).fillna(audit.status)
    print(f"{label}: {len(kept)} unique, context-compatible records")
    return kept, audit

positive_pool, positive_audit = unique_usable(positive_path, POSITIVE_MODE, "Positive")
positive_idrs = set(map(idr_sequence, positive_pool))
positives = positive_pool
if MAX_POSITIVE is not None and len(positives) > MAX_POSITIVE:
    chosen = np.random.default_rng(SEED).choice(len(positives), MAX_POSITIVE, replace=False)
    positives = [positives[i] for i in sorted(chosen)]
background_pool, background_audit = unique_usable(bg_path, BACKGROUND_MODE, "Background")
overlapping = {r.accession for r in background_pool if idr_sequence(r) in positive_idrs}
background_audit.loc[background_audit.record_id.isin(overlapping), "status"] = "overlaps positive set"
background_pool = [r for r in background_pool if idr_sequence(r) not in positive_idrs]
if not positives or not background_pool:
    raise ValueError("Need nonempty positive and nonoverlapping background sets.")
background = length_match(positives, background_pool, n=MAX_BACKGROUND, rng=np.random.default_rng(SEED))
if not background:
    raise ValueError("No background records sampled.")
for label, recs, audit in [("positive", positives, positive_audit), ("background", background, background_audit)]:
    audit["selected"] = audit.record_id.isin([r.accession for r in recs])
    audit.to_csv(out / f"{label}_input_audit.csv", index=False)
    sequence_index = summaries(recs, audit)
    sequence_index.insert(0, "dataset_sequence", np.arange(len(recs)))
    sequence_index.to_csv(out / f"{label}_sequence_index.csv", index=False)
    write_fasta(recs, out / f"{label}_selected.fasta")
    display(audit.groupby(["status", "selected"]).size().rename("records"))
print(f"Analyzing {len(positives)} positives and {len(background)} background records")


## Check background matching

Matching is approximate if the background pool cannot fill a length bin. Inspect these distributions
before interpreting enrichment; an unsuitable pool is not fixed by increasing the requested sample.
The exported selected FASTAs and audit tables record exactly which sequences were analyzed.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
lengths = [[len(idr_sequence(r)) for r in group] for group in (positives, background)]
bins = np.histogram_bin_edges(lengths[0] + lengths[1], bins=20)
for values, label in zip(lengths, ("Positive", "Background")):
    ax.hist(values, bins=bins, density=True, histtype="step", label=label)
ax.set(xlabel="IDR length (residues)", ylabel="Density")
ax.legend(frameon=False)
fig.savefig(out / "background_lengths.png", dpi=160)
plt.show()


## 2. Build feature datasets

Encode both sets once and reuse the saved sparse activations for counts and sequence logos.
Reduce `BATCH_SIZE` if GPU memory is limited. Rerunning this cell replaces the dataset files.


In [ ]:
pos_fd = sae.build_feature_dataset(positives, out / "fd_positive", batch_size=BATCH_SIZE)
bg_fd = sae.build_feature_dataset(background, out / "fd_background", batch_size=BATCH_SIZE)
print("done")


## 3. Compare feature prevalence

Here a feature is counted once per sequence if any stored activation is strictly positive.
`enrich` computes smoothed log2 odds ratios and a standardized count statistic under a
hypergeometric null, then uses normal-approximation p-values with Benjamini–Hochberg correction.
Only features with sufficient pooled counts are tested.


In [ ]:
a, n_pos = feature_counts(pos_fd)
b, n_neg = feature_counts(bg_fd)
result = enrich(a, n_pos, b, n_neg, sae.sae.num_latents)
mask = enriched_mask(result)

print(f"{int(mask.sum())} enriched features")
print(f"  FDR < {FDR_ALPHA}, log2 odds ratio >= {LOG2OR_FLOOR}, prevalence >= {PREV_POS_FLOOR:.0%}")


In [ ]:
# Export every feature, including untested features, rather than only selected hits.
print("Result fields:", list(result))
table = pd.DataFrame({key: value for key, value in result.items()
                      if isinstance(value, np.ndarray) and value.shape == (sae.sae.num_latents,)})
table.insert(0, "feature_id", np.arange(sae.sae.num_latents))
table["positive_count"] = a
table["background_count"] = b
table["enriched"] = mask
table.to_csv(out / "enrichment.csv", index=False)
display(table.loc[table.enriched].sort_values("log2or", ascending=False).head(20))


In [ ]:
import matplotlib.pyplot as plt

active = result["active"]
x, y = result["log2or"][active], np.abs(result["z"][active])
enr = mask[active]

fig, ax = plt.subplots(figsize=(4.6, 3.6), constrained_layout=True)
ax.scatter(x[~enr], y[~enr], s=4, alpha=0.25, lw=0, color="#c3ced0", label="other")
ax.scatter(x[enr], y[enr], s=8, alpha=0.9, lw=0, color="#c1440e", label="enriched")
ax.axvline(LOG2OR_FLOOR, ls="--", lw=0.8, color="#110d1b")
ax.axvline(0, lw=0.8, color="#110d1b")
ax.set_xlabel("log$_2$ odds ratio")
ax.set_ylabel("|z|")
ax.set_title(f"{NAME}: {int(mask.sum())} enriched features", fontsize=10)
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(out / "enrichment.png", dpi=160)
plt.show()


## Select features for inspection

Rank passing features by log2 odds ratio and remove features whose strongest background
activations cluster near IDR boundaries. This is a heuristic, not proof of a motif or function.
No passing features is a valid outcome; the complete results remain in `enrichment.csv`.


In [ ]:
ids = top_features(result, n=TOP_N, drop_boundary=True, feature_dir=bg_fd)
print("Selected features:", ids)
(out / "selected_features.json").write_text(json.dumps(ids))


## 5. Inspect sequence windows

Up to `N_FEATURES` selected features appear in a three-column grid, ordered left to right,
then top to bottom. For each feature, take up to `N_WINDOWS` positive sequences, with one window centered
on each sequence's strongest activation. Missing residues at sequence ends contribute no counts;
windows stay aligned to the peak.

Following the manuscript style, letter heights show information content (bits, relative to a
uniform 20-amino-acid background), chemistry sets the colors, and orange shading shows the mean
activation at each offset, normalized to the profile maximum. The dashed line marks the peak.
Each logo has its own vertical scale. These small-sample logos describe the selected windows;
they do not establish function or reproduce the manuscript's corpus-wide analysis.


In [ ]:
import logomaker
import pandas as pd

positive_dataset = FeatureDataset(pos_fd)
show = ids[:N_FEATURES]
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
aa_index = {aa: i for i, aa in enumerate(amino_acids)}
offsets = np.arange(-HALF_WIDTH, HALF_WIDTH + 1)

if not show:
    print("No signature features to plot.")
else:
    n_columns = 3
    n_rows = (len(show) + n_columns - 1) // n_columns
    fig, axes = plt.subplots(
        n_rows, n_columns, figsize=(12, 1.6 * n_rows), squeeze=False,
    )
    for ax, feature_id in zip(axes.flat, show):
        sequence_ids, _ = positive_dataset.top_sequences(
            feature_id, n=positive_dataset.n_seqs, sort_by="peak",
        )
        counts = np.zeros((len(offsets), len(amino_acids)))
        profile = np.zeros(len(offsets))
        n_windows = 0
        for seq_id in sequence_ids:
            positions, values = positive_dataset.trace(int(seq_id), feature_id)
            if not len(positions) or values.max() <= 0:
                continue
            fim_string = positive_dataset.sequence(int(seq_id))
            peak_position = int(positions[values.argmax()])
            activation_at = dict(zip(positions, values))
            for column, offset in enumerate(offsets):
                position = peak_position + int(offset)
                if position not in activation_at:
                    continue
                residue = fim_string[position]
                if residue in aa_index:
                    counts[column, aa_index[residue]] += 1
                    profile[column] += activation_at[position]
            n_windows += 1
            if n_windows >= N_WINDOWS:
                break
        if n_windows < 2:
            ax.set_axis_off()
            ax.set_title(f"F{feature_id}: too few windows")
            continue

        # Match the manuscript: empirical probabilities, no added pseudocounts
        count_frame = pd.DataFrame(counts, columns=list(amino_acids))
        count_frame = count_frame.loc[count_frame.sum(axis=1) > 0]
        probabilities = count_frame.div(count_frame.sum(axis=1), axis=0)
        information = logomaker.transform_matrix(
            probabilities, from_type="probability", to_type="information",
        )
        profile /= n_windows
        profile /= profile.max()
        for column, activation in enumerate(profile):
            if activation > 0:
                ax.axvspan(
                    column - 0.5, column + 0.5, color=(1.0, 140 / 255, 0.0),
                    alpha=float(activation) ** 0.55 * 0.65, lw=0, zorder=0,
                )
        logomaker.Logo(information, ax=ax, color_scheme="chemistry")
        ax.axvline(HALF_WIDTH, color="black", ls="--", lw=0.5, zorder=0)
        ax.text(
            0.04, 0.95, f"F{feature_id} · {n_windows} windows",
            transform=ax.transAxes, va="top", ha="left", fontsize=9,
        )
        ax.set_ylim(0, ax.get_ylim()[1] * 1.28)
        ax.set_xlim(-0.5, len(offsets) - 0.5)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes.flat[len(show):]:
        ax.set_axis_off()
    fig.suptitle(NAME.replace("_", " ").capitalize(), fontsize=12, y=0.985)
    fig.subplots_adjust(
        left=0.025, right=0.985, top=0.94, bottom=0.025, hspace=0.26, wspace=0.12,
    )
    fig.savefig(out / "feature_logos.png", dpi=160)
    plt.show()


## Export settings and optionally a training signature

Analysis is complete without training. Enable `EXPORT_SIGNATURE` only if you want a GRPO handoff.
Use the same SAE weights when consuming the feature IDs. If no features pass, no new signature
is produced; use a separate output directory for each analysis to avoid mixing runs.


In [ ]:
sig_path = None
if EXPORT_SIGNATURE and ids:
    sig_path = write_signature(out / "signature.json", {NAME: ids}, case=CASE,
                               provenance=dict(sae=SAE, positive=str(positive_path), background=str(bg_path),
                                               n_pos=len(positives), n_background=len(background), seed=SEED,
                                               length_matched=True, deduplicated=True, boundary_dropped=True))
    print(f"Signature: {sig_path}; SIGNATURE={NAME}; CASE={CASE}")
save_run(out, dict(device=str(sae.device), positive=str(positive_path), background=str(bg_path), positive_mode=POSITIVE_MODE,
                   background_mode=BACKGROUND_MODE, sae=SAE, max_positive=MAX_POSITIVE,
                   max_background=MAX_BACKGROUND, seed=SEED, batch_size=BATCH_SIZE, top_n=TOP_N,
                   name=NAME, case=CASE, export_signature=EXPORT_SIGNATURE,
                   fdr_alpha=FDR_ALPHA, log2or_floor=LOG2OR_FLOOR, prevalence_floor=PREV_POS_FLOOR), elapsed=time.perf_counter() - started)
print(f"Elapsed including model load: {time.perf_counter() - started:.1f} s")


For an optional training handoff, configure `FEATURES`, `SIGNATURE`, and `CASE` in
[the SAE GRPO script](../scripts/training/grpo/sae_features.bash). Training is a separate GPU job.
For sequence-level follow-up, open [inspect_sae_features.ipynb](inspect_sae_features.ipynb).
